# **Basic Tasks**

In [0]:
orders_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load("/Volumes/cyntexa_dev/sales/sales_volume/orders.csv")

In [0]:
orders_df.printSchema()

In [0]:
from pyspark.sql.functions import col, sum

In [0]:
null_counts = orders_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in orders_df.columns
])

In [0]:
null_counts.display()

I have null containing column is Customer Email, Region, total amount.

In [0]:
missing_email_df = orders_df.filter(col("Customer Email").isNull())
missing_region_df = orders_df.filter(col("Region").isNull())
missing_total_amount_df = orders_df.filter(col("Total Amount").isNull())

In [0]:
missing_email_df.display()

In [0]:
missing_region_df.display()

In [0]:
missing_total_amount_df.display()

In [0]:
print("Original row count:", orders_df.count())

In [0]:
distinct_df = orders_df.distinct()

print("Distinct row count:", distinct_df.count())

In [0]:
deduplicated_df = orders_df.dropDuplicates()

print("Original row count:", orders_df.count())
print("After dropDuplicates:", deduplicated_df.count())

In [0]:
from pyspark.sql.functions import count

duplicate_order_ids = (
    orders_df
    .groupBy("Order ID")
    .agg(count("*").alias("count"))
    .filter(col("count") > 1)
)

## Task 1 — Data Quality Assessment

The e-commerce orders CSV was loaded into a PySpark DataFrame. The dataset contains several data-quality issues, including NULL values and duplicate records.

The `filter()` operation was used to identify records containing NULL values in important columns such as Customer Email, Region, and Total Amount.

The `distinct()` operation was used to determine the number of unique rows, while `dropDuplicates()` was used to remove duplicate records.

Duplicate Order IDs were also identified by grouping the data by Order ID and filtering groups with a count greater than one. This helps identify potential duplicate business records before further transformations are performed.

In [0]:
orders_renamed_df = (
    orders_df
    .withColumnRenamed("Order ID","order_id")
    .withColumnRenamed("Customer ID","customer_id")
    .withColumnRenamed("Customer Name","customer_name")
    .withColumnRenamed("Product Category","product_category")
    .withColumnRenamed("Customer Email", "customer_email")
    .withColumnRenamed("City", "city")
    .withColumnRenamed("Region", "region")
    .withColumnRenamed("Product Name", "product_name")
    .withColumnRenamed("Quantity", "quantity")
    .withColumnRenamed("Unit Price", "unit_price")
    .withColumnRenamed("Total Amount", "total_amount")
    .withColumnRenamed("Order Date", "order_date")
    .withColumnRenamed("Order Status", "order_status")
)

In [0]:
orders_renamed_df.printSchema()

## Task 2 — Column Renaming

The `withColumnRenamed()` function was used to rename the columns into consistent, lowercase, snake_case names. This improves readability and makes the columns easier to reference in PySpark transformations.

The columns `Order ID`, `Customer ID`, `Customer Name`, and `Product Category` were renamed to `order_id`, `customer_id`, `customer_name`, and `product_category`.

The columns were standardized using lowercase snake_case naming conventions to make the DataFrame easier to use in subsequent PySpark transformations.

# **Intermediate Tasks**

In [0]:
from pyspark.sql.functions import (col, trim, lower, upper, when, to_date, regexp_replace)
from pyspark.sql.functions import initcap

In [0]:
def chained_DataFrame_transformation_pipeline(orders_renamed_df):

    # Remove exact duplicate rows
    clean_orders_df = orders_renamed_df.dropDuplicates()

    # Clean whitespace
    clean_orders_df = (
    clean_orders_df
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("customer_email", trim(col("customer_email")))
    .withColumn("city", trim(col("city")))
    .withColumn("region", trim(col("region")))
    .withColumn("product_category", trim(col("product_category")))
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("order_status", trim(col("order_status")))
    )

    # Standardize casing
    clean_orders_df = (
    clean_orders_df
    .withColumn("customer_name", initcap(col("customer_name")))
    .withColumn("city", initcap(col("city")))
    .withColumn("region", initcap(col("region")))
    .withColumn("product_category", initcap(col("product_category")))
    .withColumn("product_name", initcap(col("product_name")))
    .withColumn("order_status", initcap(col("order_status")))
    )
    # For email, we want lowercase
    clean_orders_df = clean_orders_df.withColumn(
    "customer_email",
    lower(col("customer_email"))
    )
    
    # Handle NULL values

    # For missing Customer Email
    clean_orders_df = clean_orders_df.withColumn(
    "customer_email",
    when(
        col("customer_email").isNull(),
        "unknown@example.com"
    ).otherwise(col("customer_email"))
    )

    # For missing regions
    clean_orders_df = clean_orders_df.withColumn(
        "region",
        when(
            col("region").isNull(),
                "unknown"
        ).otherwise(col("region")
        )
    )
    
    # For missing city
    clean_orders_df = clean_orders_df.withColumn(
        "city",
        when(
            col("city").isNull(),
            "unknown"
        ).otherwise(col("city"))
    )

    # For missing quantity
    clean_orders_df = clean_orders_df.withColumn(
        "quantity",
        when(
            col("quantity").isNull(),
            1
        ).otherwise(col("quantity"))
        )
    
    # remove unwanted characters from total amount
    clean_orders_df = clean_orders_df.withColumn(
    "total_amount",
    regexp_replace(
        col("total_amount").cast("string"),
        "[$,]",
        ""
    ).cast("double")
    )

    # Handle missing total_amount
    clean_orders_df = clean_orders_df.withColumn(
        "total_amount",
        when(
            col("total_amount").isNull(),
            col("quantity") * col("unit_price")
        ).otherwise(col("total_amount"))
    )
    
    # Convert order_date - handle multiple formats
    from pyspark.sql.functions import coalesce, try_to_date
    clean_orders_df = clean_orders_df.withColumn(
    "order_date",
    coalesce(
        try_to_date(col("order_date"), "MM-dd-yyyy"),
        try_to_date(col("order_date"), "dd/MM/yyyy"),
        try_to_date(col("order_date"), "MM/dd/yyyy"),
        try_to_date(col("order_date"), "yyyy-MM-dd"),
        try_to_date(col("order_date"), "yyyy/MM/dd"),
        try_to_date(col("order_date"), "dd-MM-yyyy")
    )
    )

    # Remove duplicate Order IDs
    clean_orders_df = clean_orders_df.dropDuplicates(
    ["order_id"]
    )
    
    return clean_orders_df

In [0]:
clean_orders_df = chained_DataFrame_transformation_pipeline(orders_renamed_df)

## Task 4 — Complete Data Cleaning Pipeline

A chained PySpark DataFrame transformation pipeline was created to clean the e-commerce orders dataset.

The pipeline removes duplicate records, trims unnecessary whitespace, standardizes text casing, handles NULL values, cleans and converts the total amount to a numeric type, converts the order date to DateType, and removes duplicate Order IDs.

The cleaned DataFrame was validated by checking remaining NULL values and duplicate Order IDs.

In [0]:
clean_orders_df.display()

In [0]:
category_reference = spark.createDataFrame(
    [
        ("Electronics", "Technology", "High Value"),
        ("Furniture", "Home & Office", "High Value"),
        ("Clothing", "Fashion", "Medium Value"),
        ("Grocery", "Daily Needs", "Low Value"),
        ("Books", "Education", "Low Value")
    ],
    [
        "product_category",
        "department",
        "category_segment"
    ]
)

In [0]:
category_reference.display()

In [0]:
from pyspark.sql.functions import countDistinct, sum, round

In [0]:
category_sales = (
    clean_orders_df
    .groupBy("product_category")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        round(sum("total_amount"), 2).alias("total_revenue")
    )
)

In [0]:
category_sales.display()

In [0]:
category_report = (
    category_sales
    .join(
        category_reference,
        on="product_category",
        how="left"
    )
)

In [0]:
category_report.display()

In [0]:
category_report = category_report.withColumn(
    "average_order_value",
    round(
        col("total_revenue") / col("total_orders"),
        2
    )
)

In [0]:
final_category_report = (
    category_report
    .orderBy(col("total_revenue").desc())
)

In [0]:
unmatched_categories = (
    category_sales
    .join(
        category_reference,
        on="product_category",
        how="left_anti"
    )
)

display(unmatched_categories)

## Task 5 — Aggregation and Join

The cleaned orders DataFrame was aggregated by product category to calculate the total number of orders, total quantity sold, and total revenue.

A category reference DataFrame was created and joined with the aggregated sales data using a left join. The reference table provides additional business information such as department and category segment.

Average order value was also calculated using total revenue divided by the total number of orders. The final report was sorted by total revenue in descending order.

A left-anti join was used to identify categories that did not have a matching reference record.

# **Advanced Tasks**

In [0]:
from pyspark.sql.functions import col

invalid_quantity_df = (
    clean_orders_df
    .filter(
        col("quantity").isNull() |
        (col("quantity") <= 0)
    )
)

display(invalid_quantity_df)

In [0]:
invalid_price_df = clean_orders_df.filter(col("unit_price").isNull() | (col("unit_price") <= 0))

display(invalid_price_df)

In [0]:
invalid_amount_df = clean_orders_df.filter(col("total_amount").isNull() | (col("total_amount") < 0))

display(invalid_amount_df)

In [0]:
from pyspark.sql.functions import when

validated_orders_df = (
    clean_orders_df
    .withColumn(
        "data_quality_status",
        when(
            (col("quantity") <= 0) |
            (col("unit_price") <= 0) |
            (col("total_amount") < 0),
            "Invalid"
        )
        .otherwise("Valid")
    )
)

In [0]:
display(
    validated_orders_df
    .groupBy("data_quality_status")
    .count()
)

In [0]:
final_orders_df = (
    validated_orders_df
    .filter(
        col("data_quality_status") == "Valid"
    )
    .drop("data_quality_status")
)

In [0]:
print("Before validation:", clean_orders_df.count())
print("After validation:", final_orders_df.count())

## Task 7 — Additional Data Quality Validation

An additional business-rule validation was implemented for the cleaned e-commerce orders dataset.

The pipeline checks whether quantity and unit price are greater than zero and whether total amount is non-negative. Records violating these business rules are marked as Invalid.

A validation flag was created to distinguish valid and invalid records. Invalid records were excluded from the final analysis DataFrame.

An additional consistency check was also performed by comparing Total Amount with Quantity multiplied by Unit Price.

# Task 8 — Git Branching Strategy

## Objective

Design a Git branching strategy for safely promoting code from development to the main production branch.

## Branch Structure

The project uses three types of branches:

- `feature/*` — Used for developing individual features or changes.
- `dev` — Used for integrating and testing completed features.
- `main` — Contains production-ready and approved code.

## Development Workflow

The development workflow is:

`feature/* → dev → main`

### 1. Feature Development

A developer creates a feature branch from the `dev` branch.

Example:

`feature/assignment5-task8`

The developer implements and tests the required changes in this branch.

### 2. Pull Request to Dev

After completing the development and testing, the developer creates a Pull Request from the feature branch to `dev`.

The changes are reviewed before being merged.

### 3. Testing in Dev

After the Pull Request is approved and merged into `dev`, the changes are tested in the development environment.

Data transformations, data quality checks, and job execution are validated.

### 4. Pull Request to Main

After successful testing in `dev`, a Pull Request is created from `dev` to `main`.

The changes are reviewed and approved before merging.

### 5. Production

The `main` branch contains the production-ready version of the project.
## 
Only tested and approved changes should be merged into `main`.

## Summary
This branching strategy keeps development work isolated in feature branches, allows integration and testing in dev, and ensures that only reviewed and tested code reaches main.

In [0]:
from pyspark.sql.functions import sum, countDistinct, round

category_revenue = (
    final_orders_df
    .groupBy("product_category")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("quantity").alias("units_sold"),
        round(sum("total_amount"), 2).alias("total_revenue")
    )
    .orderBy("total_revenue", ascending=False)
)

In [0]:
category_revenue.display()

In [0]:
region_revenue = final_orders_df.groupBy("region").agg(
    countDistinct("order_id").alias("total_orders"),
    sum("quantity").alias("units_sold"),
    round(sum("total_amount"), 2).alias("total_revenue")
    ).orderBy("total_revenue", ascending=False)

In [0]:
region_revenue.display()

In [0]:
from pyspark.sql.functions import date_format

monthly_revenue = (
    final_orders_df
    .withColumn(
        "month",
        date_format("order_date", "yyyy-MM")
    )
    .groupBy("month")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        round(sum("total_amount"), 2).alias("total_revenue")
    )
    .orderBy("month")
)

display(monthly_revenue)

In [0]:
top_products = (
    final_orders_df
    .groupBy("product_name")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("quantity").alias("units_sold"),
        round(sum("total_amount"), 2).alias("total_revenue")
    )
    .orderBy("total_revenue", ascending=False)
    .limit(10)
)

display(top_products)

### Create a Business Summary

In [0]:
top_category = category_revenue.first()

print(
    "Top Revenue Category:",
    top_category["product_category"],
    "| Revenue:",
    top_category["total_revenue"]
)

In [0]:
top_region = region_revenue.first()

print(
    "Top Revenue Region:",
    top_region["region"],
    "| Revenue:",
    top_region["total_revenue"]
)

In [0]:
top_month = (
    monthly_revenue
    .orderBy("total_revenue", ascending=False)
    .first()
)

print(
    "Highest Revenue Month:",
    top_month["month"],
    "| Revenue:",
    top_month["total_revenue"]
)

# Task 9 — Business Summary Report

## Objective

The cleaned e-commerce dataset was analyzed to answer key business questions using PySpark DataFrame transformations and aggregations.

## Business Question 1 — Which Product Category Generates the Most Revenue?

Revenue was calculated by grouping the cleaned orders by `product_category` and summing `total_amount`.

The category with the highest total revenue is identified as the top-performing product category.

## Business Question 2 — Which Region Generates the Most Revenue?

Revenue was calculated by grouping the cleaned orders by `region`.

The region with the highest total revenue is identified as the best-performing region.

## Business Question 3 — What Is the Monthly Revenue Trend?

A month column was derived from `order_date` using `date_format()`.

Monthly revenue was calculated by grouping the data by month and summing `total_amount`. This report can be used to identify revenue trends over time.

## Additional Analysis

A Top 10 product report was created by calculating total revenue and units sold for each product and ordering the results by revenue in descending order.

## Conclusion

### Business Findings

- **Top Product Category:** Electronics
- **Top Revenue Region:** unknown
- **Highest Revenue Month:** 2026-01
- **Top Product:** Laptop